In [1]:
# 1. Configuración de Spark Session                                      ║
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DateType

spark = SparkSession.builder \
    .appName("MEF_Curated_Processing_Phase3") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("✅ SparkSession creada para Fase 3")

✅ SparkSession creada para Fase 3


In [2]:
# 2. Lectura e integración de datos Parquet (2022-2025)                  
df = spark.read.parquet("Data/ANO_EJE=*/")  # Ajusta el path si es necesario
print(f"✅ Datos leídos: {df.count():,} registros, {len(df.columns)} columnas")

✅ Datos leídos: 40,173,892 registros, 62 columnas


In [3]:

# 3. Verificación y conversión de tipos de datos                         
if "FECHA" in df.columns:
    df = df.withColumn("FECHA", F.to_date("FECHA", "yyyy-MM-dd"))
    print("✅ Columna FECHA convertida a DateType")

In [4]:

#  4. Estandarización de nombres de pliego y ejecutora                    

def normalizar_texto(col):
    return F.upper(F.translate(col, "áéíóúÁÉÍÓÚ", "aeiouAEIOU"))

for col_name in ["PLIEGO_NOMBRE", "EJECUTORA_NOMBRE"]:
    if col_name in df.columns:
        df = df.withColumn(col_name, normalizar_texto(F.col(col_name)))
        print(f"✅ Columna {col_name} normalizada (mayúsculas y sin tildes)")

✅ Columna PLIEGO_NOMBRE normalizada (mayúsculas y sin tildes)
✅ Columna EJECUTORA_NOMBRE normalizada (mayúsculas y sin tildes)


In [5]:

# 5. Manejo de valores faltantes en montos                               


for col_name in ["MONTO_DEVENGADO", "MONTO_PIM", "MONTO_CERTIFICADO"]:
    if col_name in df.columns:
        df = df.withColumn(col_name, F.when(F.col(col_name).isNull(), 0).otherwise(F.col(col_name)))
        print(f"✅ Valores nulos en {col_name} reemplazados por 0")

✅ Valores nulos en MONTO_DEVENGADO reemplazados por 0
✅ Valores nulos en MONTO_PIM reemplazados por 0
✅ Valores nulos en MONTO_CERTIFICADO reemplazados por 0


In [6]:

#  6. Eliminación de registros inconsistentes                             


if "MONTO_PIM" in df.columns and "MONTO_DEVENGADO" in df.columns:
    registros_inconsistentes = df.filter((F.col("MONTO_PIM") == 0) & (F.col("MONTO_DEVENGADO") > 0)).count()
    df = df.filter(~((F.col("MONTO_PIM") == 0) & (F.col("MONTO_DEVENGADO") > 0)))
    print(f"✅ Registros inconsistentes eliminados: {registros_inconsistentes}")

✅ Registros inconsistentes eliminados: 18193945


In [7]:

#  7. Creación de columna de control para trazabilidad                    
if "ANO_EJE" in df.columns and "MES_EJE" in df.columns:
    df = df.withColumn("ANIO_MES", F.concat_ws("-", F.col("ANO_EJE"), F.col("MES_EJE")))
    print("✅ Columna de control ANIO_MES creada")

In [9]:

# 8. Guardado del dataset consolidado en formato Parquet                
df.write.mode("overwrite").parquet("curated/2022-2025")
print("✅ Dataset consolidado guardado en curated/2022-2025 (formato Parquet)")

✅ Dataset consolidado guardado en curated/2022-2025 (formato Parquet)


In [10]:

# 9. Resumen final de la fase                   

print("="*60)
print("RESUMEN FINAL DE FASE 3")
print("="*60)
print(f"Registros finales: {df.count():,}")
print(f"Columnas finales: {df.columns}")
print("Columnas clave normalizadas:", [c for c in ["PLIEGO_NOMBRE", "EJECUTORA_NOMBRE"] if c in df.columns])
print("Columnas de control:", [c for c in ["ANIO_MES", "FECHA"] if c in df.columns])
print("¡Fase 3 completada exitosamente!")

RESUMEN FINAL DE FASE 3
Registros finales: 21,979,947
Columnas finales: ['MES_EJE', 'NIVEL_GOBIERNO', 'NIVEL_GOBIERNO_NOMBRE', 'SECTOR', 'SECTOR_NOMBRE', 'PLIEGO', 'PLIEGO_NOMBRE', 'SEC_EJEC', 'EJECUTORA', 'EJECUTORA_NOMBRE', 'DEPARTAMENTO_EJECUTORA', 'DEPARTAMENTO_EJECUTORA_NOMBRE', 'PROVINCIA_EJECUTORA', 'PROVINCIA_EJECUTORA_NOMBRE', 'DISTRITO_EJECUTORA', 'DISTRITO_EJECUTORA_NOMBRE', 'SEC_FUNC', 'PROGRAMA_PPTO', 'PROGRAMA_PPTO_NOMBRE', 'TIPO_ACT_PROY', 'TIPO_ACT_PROY_NOMBRE', 'PRODUCTO_PROYECTO', 'PRODUCTO_PROYECTO_NOMBRE', 'ACTIVIDAD_ACCION_OBRA', 'ACTIVIDAD_ACCION_OBRA_NOMBRE', 'FUNCION', 'FUNCION_NOMBRE', 'DIVISION_FUNCIONAL', 'DIVISION_FUNCIONAL_NOMBRE', 'GRUPO_FUNCIONAL', 'GRUPO_FUNCIONAL_NOMBRE', 'META', 'FINALIDAD', 'META_NOMBRE', 'DEPARTAMENTO_META', 'DEPARTAMENTO_META_NOMBRE', 'FUENTE_FINANCIAMIENTO', 'FUENTE_FINANCIAMIENTO_NOMBRE', 'RUBRO', 'RUBRO_NOMBRE', 'TIPO_RECURSO', 'TIPO_RECURSO_NOMBRE', 'CATEGORIA_GASTO', 'CATEGORIA_GASTO_NOMBRE', 'TIPO_TRANSACCION', 'GENERICA', 'GE

# El proceso de la Fase 3 permitió consolidar, limpiar y estandarizar los datos presupuestales
# de los años 2022-2025. Se corrigieron tipos de datos, se normalizaron nombres clave,
# se manejaron valores faltantes y se eliminaron registros inconsistentes.
# El resultado es un dataset único, confiable y listo para análisis avanzado o visualización,
# almacenado en formato Parquet en la carpeta 'curated/2022-2025'.
# Este trabajo garantiza la calidad y trazabilidad de la información para futuras fases
# de analítica y toma de decisiones.